 RAG-Based Customer Support Assistant (with LangGraph & HITL)



1: Install

In [2]:
!pip install -q langchain langchain-community langgraph chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

2: Upload PDF

In [3]:
from google.colab import files
uploaded = files.upload()

Saving support.pdf to support.pdf


3: Load PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("support.pdf")
docs = loader.load()

print("Loaded:", len(docs))

Loaded: 3


4: Chunking

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 5


5: Embeddings + Vector DB

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings()

db = Chroma.from_documents(chunks, embedding_model)

retriever = db.as_retriever(search_kwargs={"k": 3})

print("Vector DB ready")

/tmp/ipykernel_2603/3495766999.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings()
/tmp/ipykernel_2603/3495766999.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_model = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/s

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB ready


6: Simple Answer Generator

In [8]:
def generate_answer(query, docs):
    if not docs:
        return "I don't know based on the knowledge base."

    context = " ".join([doc.page_content for doc in docs])

    return f"""
Answer:
{context[:300]}...

Question: {query}
"""

7: Define State

In [9]:
from typing import TypedDict, List, Any

class State(TypedDict, total=False):
    query: str
    docs: List[Any]
    answer: str
    confidence: float
    escalate: bool
    final_answer: str

8: Define Nodes


In [10]:
def retrieve_node(state: State):
    query = state.get("query", "")

    docs = retriever.invoke(query)

    return {"docs": docs}


def generate_node(state: State):
    query = state.get("query", "")
    docs = state.get("docs", [])

    answer = generate_answer(query, docs)

    confidence = 0.9 if docs else 0.3

    return {
        "answer": answer,
        "confidence": confidence
    }


def decision_node(state: State):
    return {
        "escalate": state.get("confidence", 0) < 0.5
    }


def hitl_node(state: State):
    print("\n⚠️ Escalated to Human Agent")
    human = input("Enter human response: ")

    return {"answer": human}


def output_node(state: State):
    return {"final_answer": state.get("answer", "")}

9: Build Graph

In [11]:
from langgraph.graph import StateGraph

builder = StateGraph(State)

builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("decision", decision_node)
builder.add_node("hitl", hitl_node)
builder.add_node("output", output_node)

builder.set_entry_point("retrieve")

builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "decision")

builder.add_conditional_edges(
    "decision",
    lambda state: "hitl" if state["escalate"] else "output"
)

builder.add_edge("hitl", "output")

graph = builder.compile()

print("Graph ready")

Graph ready


10: Ask Function

In [12]:
def ask(query):
    result = graph.invoke({"query": query})

    print("\n✅ Final Answer:\n")
    print(result["final_answer"])

11: Test

In [13]:
ask("What is refund policy?")
ask("How long is shipping?")
ask("random unrelated question")


✅ Final Answer:


Answer:
Customer Support Knowledge Base 
 
1. Refund Policy 
Customers can return products within 7 days of purchase. 
To initiate a refund: 
• The product must be unused and in original condition  
• Proof of purchase is required  
Refund process: 
• Refunds are approved after inspection  
• Once approved,...

Question: What is refund policy?


✅ Final Answer:


Answer:
If your order is delayed: 
• Check tracking status  
If order is lost: 
• Contact support immediately  
If damaged product received: 
• Request replacement within 48 hours  
 
8. Frequently Asked Questions (FAQs) 
Q1: How long does refund take? 
Refunds take 5–7 business days after approval. 
Q2: Ho...

Question: How long is shipping?


✅ Final Answer:


Answer:
• Digital products  
 
2. Shipping Policy 
Order processing: 
• Orders are processed within 2 business days  
Delivery time: 
• Standard delivery takes 3–5 business days  
• Delivery time may vary based on location  
Shipping charges: 
• Free